In [76]:
import psycopg2
import os
from dotenv import load_dotenv
import pandas as pd
import numpy as np
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer


In [3]:
# Load AWS Keys
load_dotenv()
user = os.environ.get('username')
pw = os.environ.get('password')

In [ ]:
# Connecting to database
RDS_HOST     = 'endgbv-database.cx62ou6gkd87.us-east-2.rds.amazonaws.com'      
RDS_PORT     = 5432
RDS_DBNAME   = 'postgres'
RDS_USER     = user
RDS_PASSWORD = pw

conn = psycopg2.connect(
    host=RDS_HOST,
    port=RDS_PORT,
    dbname=RDS_DBNAME,
    user=RDS_USER,
    password=RDS_PASSWORD
)

In [ ]:
# Get Schema
schema = pd.read_sql(
    '''
    SELECT *
    FROM INFORMATION_SCHEMA.COLUMNS
    WHERE TABLE_NAME = 'endgbv'
    ;
    ''', conn
)

schema

C:\Users\helen\AppData\Local\Temp\ipykernel_32472\2020471095.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  schema = pd.read_sql(


,table_catalog,table_schema,table_name,column_name,ordinal_position,column_default,is_nullable,data_type,character_maximum_length,character_octet_length,...,is_identity,identity_generation,identity_start,identity_increment,identity_maximum,identity_minimum,identity_cycle,is_generated,generation_expression,is_updatable
0,postgres,public,endgbv,Unemployment,15,None,YES,double precision,None,NaN,...,NO,None,None,None,None,None,NO,NEVER,None,YES
1,postgres,public,endgbv,Incident Precinct Code,3,None,YES,bigint,None,NaN,...,NO,None,None,None,None,None,NO,NEVER,None,YES
2,postgres,public,endgbv,Victim Reported Age,8,None,YES,double precision,None,NaN,...,NO,None,None,None,None,None,NO,NEVER,None,YES
3,postgres,public,endgbv,COMMDIST,12,None,YES,double precision,None,NaN,...,NO,None,None,None,None,None,NO,NEVER,None,YES
4,postgres,public,endgbv,Poverty,13,None,YES,double precision,None,NaN,...,NO,None,None,None,None,None,NO,NEVER,None,YES
5,postgres,public,endgbv,Median Income,14,None,YES,double precision,None,NaN,...,NO,None,None,None,None,None,NO,NEVER,None,YES
6,postgres,public,endgbv,Victim Sex,7,None,YES,text,None,1.073742e+09,...,NO,None,None,None,None,None,NO,NEVER,None,YES
7,postgres,public,endgbv,Offense Type,1,None,YES,text,None,1.073742e+09,...,NO,None,None,None,None,None,NO,NEVER,None,YES
8,postgres,public,endgbv,Suspect Race,9,None,YES,text,None,1.073742e+09,...,NO,None,None,None,None,None,NO,NEVER,None,YES
9,postgres,public,endgbv,Suspect Sex,10,None,YES,text,None,1.073742e+09,...,NO,None,None,None,None,None,NO,NEVER,None,YES


In [52]:
# Retrieve Data
df = pd.read_sql(
    '''
    SELECT
        "Offense Type", "Report Date", "Incident Precinct Code",
        "Borough Name", "Victim Reported Age", "Suspect Reported Age",
        "COMMDIST",
        CASE
            WHEN "Unemployment" = 1 THEN 'HIGH'
            ELSE 'LOW'
        END AS "Unemployment",
        CASE
            WHEN "Median Income" = 1 THEN 'HIGH'
            ELSE 'LOW'
        END AS "Median Income",
        CASE
            WHEN "Poverty" = 1 THEN 'HIGH'
            ELSE 'LOW'
        END AS "Poverty",
        CASE 
            WHEN "Intimate Relationship Flag" = 'NO' THEN 'No'
            WHEN "Intimate Relationship Flag" = 'YES' THEN 'Yes'
            WHEN "Intimate Relationship Flag" = 'MISSING' THEN NULL
            WHEN "Intimate Relationship Flag" IS NULL THEN NULL
            ELSE "Intimate Relationship Flag"
        END AS "Intimate Relationship Flag",
        CASE
            WHEN "Victim Race" = 'None' THEN NULL
            WHEN "Victim Race" IS NULL THEN NULL
            ELSE "Victim Race"
        END AS "Victim Race",
        CASE 
            WHEN "Victim Sex" = 'None' THEN NULL
            WHEN "Victim Sex" IS NULL THEN NULL
            ELSE "Victim Sex"
        END AS "Victim Sex",
        CASE
            WHEN "Suspect Race" = 'None' THEN NULL
            WHEN "Suspect Race" IS NULL THEN NULL
            ELSE "Suspect Race"
        END AS "Suspect Race",
        CASE 
            WHEN "Suspect Sex" = 'None' THEN NULL
            WHEN "Suspect Sex" = 'UNKNWN' THEN NULL
            WHEN "Suspect Sex" IS NULL THEN NULL
            ELSE "Suspect Sex"
        END AS "Suspect Sex",
        CASE
            WHEN DATE("Report Date") > '2020-06-08' THEN 'AFTER'
            ELSE 'BEFORE'
        END AS after_reopening
    FROM endgbv
    ;
    ''', conn
)

C:\Users\helen\AppData\Local\Temp\ipykernel_32472\2552974733.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


In [58]:
# Recode NAs
df = df.replace({None: np.nan})

In [73]:
# Check Missingness
missingness_df = df.isna().mean().reset_index(name="pct_missing").assign(more_than_25 = lambda x: x["pct_missing"] > 0.25)

missingness_df


,index,pct_missing,more_than_25
0,Offense Type,0.000000,False
1,Report Date,0.000000,False
2,Incident Precinct Code,0.000000,False
3,Borough Name,0.000000,False
4,Victim Reported Age,0.287865,True
5,Suspect Reported Age,0.378238,True
6,COMMDIST,0.000215,False
7,Unemployment,0.000000,False
8,Median Income,0.000000,False
9,Poverty,0.000000,False


In [75]:
temp = df.drop(columns = missingness_col[missingness_col["more_than_25"] == True]["index"].tolist())

In [77]:
temp

,Offense Type,Report Date,Incident Precinct Code,Borough Name,COMMDIST,Unemployment,Median Income,Poverty,Intimate Relationship Flag,Victim Race,Victim Sex,Suspect Race,Suspect Sex,after_reopening
0,DIR,12/31/2021,40,BRONX,201.0,HIGH,HIGH,HIGH,Yes,WHITE,FEMALE,WHITE,MALE,AFTER
1,DIR,12/31/2021,40,BRONX,201.0,HIGH,HIGH,HIGH,NaN,NaN,NaN,NaN,NaN,AFTER
2,DIR,12/31/2021,40,BRONX,201.0,HIGH,HIGH,HIGH,No,BLACK,FEMALE,BLACK,MALE,AFTER
3,DIR,12/31/2021,40,BRONX,201.0,HIGH,HIGH,HIGH,NaN,NaN,NaN,NaN,NaN,AFTER
4,DIR,12/31/2021,40,BRONX,201.0,HIGH,HIGH,HIGH,No,BLACK,FEMALE,BLACK,FEMALE,AFTER
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
483785,DIR,01/01/2020,123,STATEN ISLAND,503.0,LOW,LOW,LOW,No,WHITE,MALE,WHITE,FEMALE,BEFORE
483786,DIR,01/01/2020,122,STATEN ISLAND,595.0,LOW,LOW,LOW,NaN,NaN,NaN,NaN,NaN,BEFORE
483787,DIR,01/01/2020,122,STATEN ISLAND,595.0,LOW,LOW,LOW,Yes,ASIAN / PACIFIC ISLANDER,FEMALE,WHITE,MALE,BEFORE
483788,FELONY ASSAULT,12/31/2019,75,BROOKLYN,305.0,LOW,HIGH,HIGH,Yes,WHITE,FEMALE,BLACK,MALE,BEFORE


In [78]:
conn.close()
print("Connection closed.")

Connection closed.
